# JAX JIT Analysis
Homework 3 – JIT Compilation and Tracing for ML on GPUs

## Setup

In [ ]:

import jax
import jax.numpy as jnp
import numpy as np
import time
import matplotlib.pyplot as plt


## Part 1: Measuring Compilation Overhead

In [ ]:

def elementwise_ops(x):
    y = jnp.sin(x)
    y = jnp.cos(y)
    y = jnp.exp(y)
    y = jnp.log(y + 1)
    y = jnp.square(y)
    y = jnp.sin(y)
    y = jnp.cos(y)
    y = jnp.exp(y)
    y = jnp.log(y + 1)
    y = jnp.square(y)
    return y

jit_fn = jax.jit(elementwise_ops)

sizes = [100, 500, 1000, 5000]

eager_times = []
jit_first = []
jit_second = []

for s in sizes:
    x = jnp.ones((s, s))

    start = time.time()
    elementwise_ops(x).block_until_ready()
    eager_times.append(time.time() - start)

    start = time.time()
    jit_fn(x).block_until_ready()
    jit_first.append(time.time() - start)

    start = time.time()
    jit_fn(x).block_until_ready()
    jit_second.append(time.time() - start)

print("Eager:", eager_times)
print("JIT first:", jit_first)
print("JIT second:", jit_second)


In [ ]:

plt.figure()
plt.plot(sizes, eager_times, marker="o", label="Eager")
plt.plot(sizes, jit_first, marker="o", label="JIT First Call")
plt.plot(sizes, jit_second, marker="o", label="JIT Cached")

plt.xlabel("Matrix Size")
plt.ylabel("Execution Time (seconds)")
plt.title("Execution Time vs Matrix Size")
plt.legend()
plt.show()



### Explanation

The first JIT call includes compilation time. Smaller matrices show higher overhead because the compile time dominates the compute time. For larger matrices, computation dominates and the compiled version becomes faster.


## Part 2: Shape Specialization

In [ ]:

@jax.jit
def row_mean(x):
    return jnp.mean(x, axis=1)

shapes = [(100,100), (100,200), (100,100), (200,100)]
times = []

for shape in shapes:
    x = jnp.ones(shape)

    start = time.time()
    row_mean(x).block_until_ready()
    times.append(time.time() - start)

print(times)


In [ ]:

plt.figure()
plt.bar(range(len(times)), times)
plt.xticks(range(len(times)), [str(s) for s in shapes])
plt.ylabel("Execution Time")
plt.title("Retracing Cost for Different Shapes")
plt.show()



### Discussion

JAX specializes compiled functions to specific tensor shapes. When a new shape is used, JAX retraces and recompiles the function, which increases execution time for that call.


## Part 3: Operator Fusion

In [ ]:

def version_a(x):
    result = 0
    for i in range(1,101):
        result += jnp.sin(x) + jnp.cos(x)
    return result

@jax.jit
def version_b(x):
    result = 0
    for i in range(1,101):
        result += jnp.sin(x) + jnp.cos(x)
    return result

x = jnp.ones((2000,2000))

start = time.time()
version_a(x).block_until_ready()
time_a = time.time() - start

start = time.time()
version_b(x).block_until_ready()
time_b = time.time() - start

print("Version A:", time_a)
print("Version B:", time_b)
